# PKG Geo — MDM Address Profiling & Extract (v2)

**Source:** `dsihd01p_dsi.neo4j_address`
**Target:** node-level geo CSV consumed by the PKG metric modules

---

## What changed from v1, and why

| | v1 assumed | Run showed | Consequence |
|---|---|---|---|
| Grain | many addresses per party | **`n_rows == n_parties`** in every snapshot | one address per party. Section F and `addr_rank` deleted. |
| History | probably one snapshot | **649 daily snapshots from 2024-09-20** | the D1 change log is buildable *retroactively*. New section F. |
| Schema | flags/dates likely hidden off-screen | 18 columns, no active flag, no effective dates, no quality code | `addr_cleansed_date` + `coord_decimals` are the only quality signals. |
| Compute | `persist(MEMORY_AND_DISK)` on the raw snapshot | `FetchFailedException`, 2 GB executors | wide persist removed; narrow parquet materialisation added. |

### The compute fix, stated once

The table now holds roughly **17 billion rows** (649 × 26.8M). Every profiling pass
that touches `spark.table(ADDR_TABLE)` risks re-scanning it. Section 0.3 writes a
narrow projection of the latest snapshot to parquet **once**; everything downstream
reads that. Rules applied throughout:

- never persist a wide DataFrame — project first, persist narrow, prefer `DISK_ONLY`
- `approx_count_distinct` in profiling contexts; exact counts only where correctness
  depends on them
- no `.collect()` on anything unbounded

### Populations, unchanged from v1

This table is **all customers**, not the PKG universe. Every coverage number is
reported against `MDM_ALL`, `PKG_NODE`, and `PKG_DOLLAR`. The Pittsburgh belief gets
tested on the third.

In [ ]:
import json
import os
import re
from datetime import datetime

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

## 0. Config

In [ ]:
# ---- Tables -----------------------------------------------------------------
ADDR_TABLE = "dsihd01p_dsi.neo4j_address"
PKG_METRICS_TABLE = "bdahd01p_dlcdi1_cdi_tm.cust_c2c_metrics"
PKG_ROLES_TABLE = "bdahd01p_dlcdi1_cdi_tm.cust_c2c_roles"

# ---- Scratch space for materialised snapshots -------------------------------
# REQUIRED. A path Spark can write to (HDFS, or a shared mount visible to executors).
# If left None the notebook still runs, but every section re-scans the source table.
WORK_DIR = None                  # e.g. "hdfs:///user/sa15474/pkg/geo"
OUT_DIR_LOCAL = "../metrics/geo"  # driver-local, for the CSV handoff

# ---- Columns kept in the narrow snapshot ------------------------------------
NARROW_COLS = [
    "mdm_id", "mdm_address_id", "addr_type", "addr_loc_rec_type",
    "latitude_degrees", "longitude_degrees",
    "city", "state_or_province", "zip_cd", "addr_country",
    "addr_line_1", "addr_line_2", "addr_cleansed_date",
]

# ---- PKG time window --------------------------------------------------------
PKG_TIME_COL = None
PKG_TIME_MIN = None
PKG_TIME_MAX = None

# ---- Change log (section F) -------------------------------------------------
CHANGE_LOG_SNAPSHOTS = "month_end"   # "month_end" | "quarter_end" | "endpoints"
CHANGE_LOG_MAX_SNAPS = 26            # hard cap; each one is a 26.8M-row read

# ---- Thresholds -------------------------------------------------------------
DUP_MIN_PARTIES = 25
DUP_ROUND_DP = 5
LOW_PRECISION_DP = 2      # coords with <= this many decimals are ~1 km: centroid
TOP_N = 60
APPROX_RSD = 0.01

PIT_LAT, PIT_LON = 40.4406, -79.9959

REPORT = {"generated_at": datetime.now().isoformat(timespec="seconds"), "version": 2}


def note(key, value):
    REPORT[key] = value
    print(f"  {key}: {value}")


def show(df, n=TOP_N, truncate=False):
    df.show(n, truncate=truncate)

In [ ]:
spark = (
    SparkSession.builder.appName("pkg_geo_address_profile_v2")
    # Executors are small (~2 GB observed). Smaller tasks + AQE coalescing is the
    # right trade: many cheap tasks rather than few that die and trigger FetchFailed.
    .config("spark.sql.shuffle.partitions", "800")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.shuffle.io.maxRetries", "10")
    .config("spark.shuffle.io.retryWait", "15s")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .enableHiveSupport()
    .getOrCreate()
)
print("Spark", spark.version)

try:
    _ = F.xxhash64
    def stable_hash(*cols):
        return F.xxhash64(*cols)
except AttributeError:
    def stable_hash(*cols):
        return F.substring(F.sha2(F.concat_ws("|", *cols), 256), 1, 16)

### 0.1 Is the source table partitioned by `load_dt`?

This determines whether a single-snapshot filter reads 26.8M rows or 17 billion. If
it is not partitioned, `WORK_DIR` stops being an optimisation and becomes mandatory.

In [ ]:
try:
    parts = spark.sql(f"SHOW PARTITIONS {ADDR_TABLE}")
    n_parts = parts.count()
    print(f"partitioned: {n_parts} partitions")
    show(parts.orderBy(F.desc(parts.columns[0])).limit(5), 5)
    IS_PARTITIONED = True
except Exception as e:
    print("SHOW PARTITIONS failed — table is probably NOT partitioned:")
    print("   ", str(e).splitlines()[0][:200])
    IS_PARTITIONED = False

note("P0_is_partitioned", IS_PARTITIONED)
if not IS_PARTITIONED and WORK_DIR is None:
    print("\n  !! Unpartitioned source AND no WORK_DIR. Every cell below will scan the "
          "full history. Set WORK_DIR before continuing.")

### 0.2 Snapshot inventory

Cached, because section A reads its grain numbers straight off this rather than
re-aggregating 26.8M rows.

In [ ]:
SNAP_COL = "load_dt"
addr_raw = spark.table(ADDR_TABLE)

snap_hist = (
    addr_raw.groupBy(SNAP_COL)
    .agg(F.count("*").alias("n_rows"), F.countDistinct("mdm_id").alias("n_parties"))
    .persist(StorageLevel.DISK_ONLY)          # narrow: 649 rows
)
n_snaps = snap_hist.count()
note("A0_n_snapshots", n_snaps)

snap_list = [r[SNAP_COL] for r in snap_hist.select(SNAP_COL).orderBy(SNAP_COL).collect()]
MAX_SNAP = snap_list[-1]
note("A0_snapshot_range", [snap_list[0], MAX_SNAP])
note("A0_snapshot_used", str(MAX_SNAP))

show(snap_hist.orderBy(F.desc(SNAP_COL)).limit(10), 10)

In [ ]:
# A — grain, free. n_rows == n_parties would mean one address per party.
g = snap_hist.filter(F.col(SNAP_COL) == F.lit(MAX_SNAP)).collect()[0]
note("A_n_rows", g["n_rows"])
note("A_n_parties", g["n_parties"])
ONE_ADDR_PER_PARTY = g["n_rows"] == g["n_parties"]
note("A_one_address_per_party", ONE_ADDR_PER_PARTY)

if ONE_ADDR_PER_PARTY:
    print("\n  MDM has already collapsed to one primary address per party. No selection\n"
          "  rule is needed — but B1 now matters more, because addr_type tells you which\n"
          "  KIND of address was chosen for everyone.")
else:
    print("\n  !! Multiple addresses per party in this snapshot. Restore the addr_rank\n"
          "  selection rule from v1 section H before building the node file.")

### 0.3 Materialise the narrow snapshot

The fix for the v1 crash. One read of the source table, projected to the columns
that are actually used, written to parquet. Everything below reads this.

In [ ]:
addr_snap_path = f"{WORK_DIR}/addr_snapshot_{MAX_SNAP}" if WORK_DIR else None

if addr_snap_path:
    (
        addr_raw.filter(F.col(SNAP_COL) == F.lit(MAX_SNAP))
        .select(*NARROW_COLS)
        .repartition(200)
        .write.mode("overwrite")
        .parquet(addr_snap_path)
    )
    addr = spark.read.parquet(addr_snap_path)
    print("materialised ->", addr_snap_path)
else:
    addr = addr_raw.filter(F.col(SNAP_COL) == F.lit(MAX_SNAP)).select(*NARROW_COLS)
    print("WORK_DIR unset — reading from source each time (slow)")

addr.createOrReplaceTempView("addr_latest")   # session-scoped; no DDL privilege

## Shared expressions

Defined once so the profiling and extract sections cannot drift apart.

In [ ]:
NUM_RE = r"^\s*-?\d+(\.\d+)?\s*$"
NULL_TOKENS = ["null", "none", "na", "n/a", "nan", "unknown", "-", "."]

lat_s = F.trim(F.coalesce(F.col("latitude_degrees"), F.lit("")))
lon_s = F.trim(F.coalesce(F.col("longitude_degrees"), F.lit("")))

lat_num = F.when(lat_s.rlike(NUM_RE), lat_s.cast("double"))
lon_num = F.when(lon_s.rlike(NUM_RE), lon_s.cast("double"))

has_geo = (
    lat_num.isNotNull() & lon_num.isNotNull()
    & ~((lat_num == 0) & (lon_num == 0))
    & lat_num.between(-90, 90) & lon_num.between(-180, 180)
).cast("int")

dec = lambda c: F.coalesce(F.length(F.split(F.trim(F.col(c)), r"\.").getItem(1)), F.lit(0))

u1 = F.upper(F.trim(F.coalesce(F.col("addr_line_1"), F.lit(""))))
u2 = F.upper(F.trim(F.coalesce(F.col("addr_line_2"), F.lit(""))))
rec_u = F.upper(F.trim(F.coalesce(F.col("addr_loc_rec_type"), F.lit(""))))
type_u = F.upper(F.trim(F.coalesce(F.col("addr_type"), F.lit(""))))
zip_t = F.trim(F.coalesce(F.col("zip_cd"), F.lit("")))
zip5 = F.split(zip_t, "-").getItem(0)

PO_BOX_RE = r"(^| )P ?\.? ?O ?\.? ?BOX"
REG_AGENT_RE = (
    r"(REGISTERED AGENT|CORPORATION SERVICE|CT CORPORATION|NATIONAL REGISTERED|"
    r"INCORP SERVICES|LEGALZOOM|COGENCY|VCORP|NORTHWEST REGISTERED|CSC )"
)

# Working frame: narrow, derived once, disk-cached. This is safe to persist —
# it is ~12 columns of mostly small types, not the 18-column raw row.
work = (
    addr.select(
        F.trim(F.col("mdm_id").cast("string")).alias("mdm_id"),
        F.trim(F.col("mdm_address_id").cast("string")).alias("mdm_address_id"),
        type_u.alias("addr_type"),
        rec_u.alias("rec_type"),
        lat_num.alias("lat"),
        lon_num.alias("lon"),
        has_geo.alias("has_geo"),
        F.least(dec("latitude_degrees"), dec("longitude_degrees")).alias("coord_dp"),
        zip5.alias("zip5"),
        F.when(zip_t.rlike(r"^\d{5}-\d{4}$"), F.split(zip_t, "-").getItem(1)).alias("zip4"),
        F.substring(zip5, 1, 3).alias("zip3"),
        F.upper(F.trim(F.coalesce(F.col("state_or_province"), F.lit("")))).alias("state"),
        F.upper(F.trim(F.coalesce(F.col("city"), F.lit("")))).alias("city"),
        F.upper(F.trim(F.coalesce(F.col("addr_country"), F.lit("")))).alias("country"),
        F.when(u1.rlike(PO_BOX_RE), 1).otherwise(0).alias("flag_po_box"),
        F.when(u1.rlike(REG_AGENT_RE), 1).otherwise(0).alias("flag_reg_agent"),
        F.when(u1.rlike(r"(^| )(C/O|C O |ATTN|PMB)"), 1).otherwise(0).alias("flag_care_of"),
        F.when(u1.rlike(r"(^| )PNC( |$)"), 1).otherwise(0).alias("flag_pnc"),
        F.when(u2.rlike(r"^(APT|UNIT|#|TRLR|LOT|SPC)"), "residential")
         .when(u2.rlike(r"^(STE|SUITE|FL |FLOOR|RM |DEPT|BLDG)"), "commercial")
         .otherwise(F.lit(None).cast("string")).alias("addr_unit_type"),
        stable_hash(F.concat_ws("|", F.regexp_replace(u1, "[^A-Z0-9]", ""), zip5))
            .cast("string").alias("addr_norm_hash"),
        F.col("addr_cleansed_date").cast("string").alias("addr_cleansed_date"),
    )
    .persist(StorageLevel.DISK_ONLY)
)
note("W_work_rows", work.count())

## B. `addr_type` and `addr_loc_rec_type`

With one address per party, `addr_type` no longer drives a selection rule — it tells
you what the single stored address *is*. If most parties carry a MAILING address, the
registered-point problem is worse than the rec_type mix alone suggests.

In [ ]:
b1 = (
    work.groupBy("addr_type")
    .agg(
        F.count("*").alias("n_rows"),
        F.sum("has_geo").alias("n_geocoded"),
        F.round(100 * F.avg("has_geo"), 2).alias("pct_geocoded"),
        F.round(100 * F.avg("flag_po_box"), 2).alias("pct_po_box"),
    )
    .withColumn("pct_of_book", F.round(100 * F.col("n_rows") / F.sum("n_rows").over(Window.partitionBy()), 2))
    .orderBy(F.desc("n_rows"))
)
show(b1)
REPORT["B1_addr_type"] = [r.asDict() for r in b1.collect()]

In [ ]:
b3 = (
    work.groupBy("rec_type")
    .agg(
        F.count("*").alias("n_rows"),
        F.round(100 * F.avg("has_geo"), 2).alias("pct_geocoded"),
        F.round(F.avg("coord_dp"), 2).alias("mean_coord_dp"),
    )
    .withColumn("pct_of_book", F.round(100 * F.col("n_rows") / F.sum("n_rows").over(Window.partitionBy()), 3))
    .orderBy(F.desc("n_rows"))
)
show(b3)
REPORT["B3_rec_type"] = [r.asDict() for r in b3.collect()]

b2 = (
    work.groupBy("addr_type", "rec_type")
    .agg(F.count("*").alias("n_rows"))
    .orderBy("addr_type", F.desc("n_rows"))
)
show(b2, 100)
REPORT["B2_type_x_rec"] = [r.asDict() for r in b2.collect()]

## C. Coordinate pathology

Every bucket below `1_null` survives a plain `IS NOT NULL`.
`7_positive_lon` is a dropped negative sign — fixable, not discardable.

In [ ]:
coord_state = (
    F.when(F.col("latitude_degrees").isNull() | F.col("longitude_degrees").isNull(), "1_null")
    .when((lat_s == "") | (lon_s == ""), "2_empty")
    .when(F.lower(lat_s).isin(NULL_TOKENS) | F.lower(lon_s).isin(NULL_TOKENS), "3_null_token")
    .when(~lat_s.rlike(NUM_RE) | ~lon_s.rlike(NUM_RE), "4_non_numeric")
    .when((lat_s.cast("double") == 0) & (lon_s.cast("double") == 0), "5_null_island")
    .when(~lat_s.cast("double").between(-90, 90) | ~lon_s.cast("double").between(-180, 180), "6_out_of_range")
    .when(lon_s.cast("double") > 0, "7_positive_lon_dropped_sign")
    .when(lat_s.cast("double").between(24, 50) & lon_s.cast("double").between(-125, -66), "8_ok_conus")
    .otherwise("9_ok_non_conus")
)

c1 = (
    addr.withColumn("coord_state", coord_state)
    .groupBy("coord_state")
    .agg(F.count("*").alias("n_rows"))
    .withColumn("pct", F.round(100 * F.col("n_rows") / F.sum("n_rows").over(Window.partitionBy()), 3))
    .orderBy("coord_state")
)
show(c1)
REPORT["C1_coord_state"] = [r.asDict() for r in c1.collect()]

In [ ]:
# Sample the pathological buckets — the actual strings decide what is salvageable.
show(
    addr.withColumn("coord_state", coord_state)
    .filter(F.col("coord_state").rlike("^[2-7]_"))
    .select("coord_state", "latitude_degrees", "longitude_degrees",
            "city", "state_or_province", "zip_cd", "addr_loc_rec_type")
    .limit(40),
    40,
)

In [ ]:
# C2 — decimal places. <= LOW_PRECISION_DP is ~1 km: a centroid, whatever rec_type says.
c2 = (
    work.filter(F.col("has_geo") == 1)
    .groupBy("coord_dp", "rec_type")
    .agg(F.count("*").alias("n_rows"))
    .orderBy("coord_dp", F.desc("n_rows"))
)
show(c2, 60)
REPORT["C2_coord_decimals"] = [r.asDict() for r in c2.collect()]

low_prec = work.filter((F.col("has_geo") == 1) & (F.col("coord_dp") <= LOW_PRECISION_DP)).count()
note("C2_low_precision_rows", low_prec)

## D. Placeholder detection (F1)

Conditioned on `rec_type`. Duplicates inside `HIGHRISE` are the building and are
expected. Duplicates inside `NORMAL` are registered agents, CPA firms, shared service
addresses, and PNC's own branches.

In [ ]:
coord_key = F.concat_ws(
    "_",
    F.round(F.col("lat"), DUP_ROUND_DP).cast("string"),
    F.round(F.col("lon"), DUP_ROUND_DP).cast("string"),
)

d1 = (
    work.filter(F.col("has_geo") == 1)
    .withColumn("coord_key", coord_key)
    .groupBy("rec_type", "coord_key")
    .agg(
        F.count("*").alias("n_parties"),
        F.min("city").alias("sample_city"),
        F.min("state").alias("sample_state"),
        F.max("flag_reg_agent").alias("any_reg_agent"),
        F.max("flag_pnc").alias("any_pnc"),
    )
    .filter(F.col("n_parties") >= DUP_MIN_PARTIES)
    .persist(StorageLevel.DISK_ONLY)
)
show(d1.orderBy(F.desc("n_parties")).limit(60), 60, truncate=40)
note("D1_suspect_clusters", d1.count())
note("D1_parties_on_suspect_coord",
     d1.agg(F.sum("n_parties")).collect()[0][0] or 0)
REPORT["D1_top_clusters"] = [r.asDict() for r in d1.orderBy(F.desc("n_parties")).limit(30).collect()]

In [ ]:
# D2 — shared service addresses, independent of geocode rounding
d2 = (
    work.groupBy("addr_norm_hash")
    .agg(
        F.count("*").alias("n_parties"),
        F.min("city").alias("sample_city"),
        F.min("state").alias("sample_state"),
        F.min("zip5").alias("sample_zip5"),
        F.max("flag_reg_agent").alias("any_reg_agent"),
    )
    .filter(F.col("n_parties") >= DUP_MIN_PARTIES)
    .orderBy(F.desc("n_parties"))
)
show(d2.limit(60), 60, truncate=40)
REPORT["D2_top_shared_addresses"] = [r.asDict() for r in d2.limit(30).collect()]

In [ ]:
# D3 — token flag totals
d3 = work.agg(
    F.count("*").alias("n_rows"),
    F.sum("flag_po_box").alias("po_box"),
    F.sum("flag_care_of").alias("care_of"),
    F.sum("flag_reg_agent").alias("reg_agent"),
    F.sum("flag_pnc").alias("pnc_address"),
    F.sum(F.when(F.col("addr_unit_type") == "residential", 1).otherwise(0)).alias("unit_residential"),
    F.sum(F.when(F.col("addr_unit_type") == "commercial", 1).otherwise(0)).alias("unit_commercial"),
)
show(d3, 1)
REPORT["D3_token_flags"] = d3.collect()[0].asDict()

## E. ZIP, state, country

ZIP5 is the join key to the HUD USPS crosswalk → ZCTA → county FIPS → CBSA, and it
survives when lat/lon does not — which is what makes section G6 possible.

In [ ]:
zip_shape = (
    F.when(zip_t == "", "empty")
    .when(zip_t.rlike(r"^\d{5}-\d{4}$"), "zip9_dash")
    .when(zip_t.rlike(r"^\d{9}$"), "zip9_flat")
    .when(zip_t.rlike(r"^\d{5}$"), "zip5")
    .when(zip_t.rlike(r"^\d{4}$"), "zip4_LEADING_ZERO_ALREADY_LOST")
    .when(zip_t.rlike(r"^[A-Z]\d[A-Z]"), "canada_fsa")
    .otherwise("other")
)
e1 = addr.withColumn("zip_shape", zip_shape).groupBy("zip_shape").count().orderBy(F.desc("count"))
show(e1)
REPORT["E1_zip_shape"] = [r.asDict() for r in e1.collect()]

e2 = work.groupBy("country").count().orderBy(F.desc("count"))
show(e2, 40)
REPORT["E2_country"] = [r.asDict() for r in e2.limit(40).collect()]

e3 = (
    work.groupBy("state")
    .agg(F.count("*").alias("n_parties"), F.round(100 * F.avg("has_geo"), 2).alias("pct_geocoded"))
    .orderBy(F.desc("n_parties"))
)
show(e3, 80)
REPORT["E3_state"] = [r.asDict() for r in e3.limit(80).collect()]

## F. Address change history — D1, retroactive

649 daily snapshots back to 2024-09-20 means the registered-point change log does not
have to be built going forward. It already exists.

**Two hashes, not one.** The whole point is separating data events from real moves:

| address hash | coord hash | reading |
|---|---|---|
| same | same | no change |
| same | **changed** | **re-geocode** — vendor refresh, not a relocation |
| **changed** | changed | candidate relocation |
| **changed** | same | address correction, same building |

And the daily grain gives a second, stronger discriminator for free: plot changes per
date. A vendor re-cleansing shows up as a spike affecting a large share of the book on
a single day. Real relocations are a low, flat baseline. That is `geo_change_reason`
derived from evidence rather than assumed.

In [ ]:
if CHANGE_LOG_SNAPSHOTS == "endpoints":
    sample_snaps = [snap_list[0], snap_list[-1]]
else:
    by_period = {}
    for s in snap_list:
        key = s[:7] if CHANGE_LOG_SNAPSHOTS == "month_end" else f"{s[:4]}Q{(int(s[5:7]) - 1) // 3}"
        by_period[key] = s          # last snapshot seen in each period
    sample_snaps = sorted(by_period.values())[-CHANGE_LOG_MAX_SNAPS:]

print(f"{len(sample_snaps)} snapshots: {sample_snaps[0]} .. {sample_snaps[-1]}")
note("F_snapshots_sampled", len(sample_snaps))

In [ ]:
hist_path = f"{WORK_DIR}/addr_hist" if WORK_DIR else None

hist_src = (
    addr_raw.filter(F.col(SNAP_COL).isin(sample_snaps))
    .select(
        F.col(SNAP_COL).alias("snap"),
        F.trim(F.col("mdm_id").cast("string")).alias("mdm_id"),
        stable_hash(F.concat_ws("|",
            F.regexp_replace(u1, "[^A-Z0-9]", ""),
            F.regexp_replace(u2, "[^A-Z0-9]", ""),
            zip5, F.upper(F.trim(F.coalesce(F.col("state_or_province"), F.lit("")))),
        )).alias("addr_h"),
        stable_hash(F.concat_ws("|", lat_s, lon_s)).alias("coord_h"),
        rec_u.alias("rec_type"),
    )
)

if hist_path:
    hist_src.write.mode("overwrite").partitionBy("snap").parquet(hist_path)
    hist = spark.read.parquet(hist_path)
else:
    hist = hist_src
    print("WORK_DIR unset — the change log will re-read the source. Expect this to be slow.")

In [ ]:
w = Window.partitionBy("mdm_id").orderBy("snap")

changes = (
    hist.withColumn("prev_addr_h", F.lag("addr_h").over(w))
    .withColumn("prev_coord_h", F.lag("coord_h").over(w))
    .filter(F.col("prev_addr_h").isNotNull())
    .withColumn(
        "change_kind",
        F.when((F.col("addr_h") == F.col("prev_addr_h")) & (F.col("coord_h") == F.col("prev_coord_h")), "none")
        .when(F.col("addr_h") == F.col("prev_addr_h"), "regeocode")
        .when(F.col("coord_h") == F.col("prev_coord_h"), "addr_correction_same_point")
        .otherwise("candidate_relocation"),
    )
)

f2 = (
    changes.groupBy("snap", "change_kind")
    .agg(F.count("*").alias("n_parties"))
    .groupBy("snap")
    .pivot("change_kind")
    .agg(F.first("n_parties"))
    .orderBy("snap")
)
show(f2, 40)
REPORT["F2_changes_by_snapshot"] = [r.asDict() for r in f2.collect()]

**Reading F2.** Look down the `regeocode` column first. Any row where it jumps by
orders of magnitude is a vendor refresh date — every "relocation" adjacent to it is
suspect and those dates should be excluded from the dynamics module, not modelled.
The `candidate_relocation` baseline, net of those dates, is the real churn rate, and
it sets expectations for how much signal geo dynamics can carry at all.

In [ ]:
# F3 — parties that ever changed, and how often. The tail is the interesting part:
# a party with many address changes in 22 months is either a data-quality victim or
# genuinely itinerant, and both are worth a flag.
churn = (
    changes.filter(F.col("change_kind") != "none")
    .groupBy("mdm_id")
    .agg(
        F.count("*").alias("n_changes"),
        F.sum(F.when(F.col("change_kind") == "candidate_relocation", 1).otherwise(0)).alias("n_reloc"),
    )
)
f3 = churn.groupBy("n_changes").agg(F.count("*").alias("n_parties")).orderBy("n_changes")
show(f3, 30)
REPORT["F3_change_frequency"] = [r.asDict() for r in f3.limit(30).collect()]
note("F3_parties_ever_changed", churn.count())

## G. PKG join and three-population coverage

The blocker section. Everything above is data quality; this decides whether the geo
module is buildable.

In [ ]:
pkg_raw = spark.table(PKG_METRICS_TABLE)
pkg_raw.printSchema()

if PKG_TIME_COL is None:
    TIME_CANDIDATES = ["time_key", "month_key", "yyyymm", "period", "as_of_month", "rpt_month", "load_dt"]
    PKG_TIME_COL = next((c for c in TIME_CANDIDATES if c in pkg_raw.columns), None)
print("PKG time column:", PKG_TIME_COL)

pkg = pkg_raw
if PKG_TIME_COL and PKG_TIME_MIN:
    pkg = pkg.filter(F.col(PKG_TIME_COL) >= F.lit(PKG_TIME_MIN))
if PKG_TIME_COL and PKG_TIME_MAX:
    pkg = pkg.filter(F.col(PKG_TIME_COL) <= F.lit(PKG_TIME_MAX))

In [ ]:
NAICS_PLACEHOLDER_RE = r"^\s*(\*+|-1|0+|UNK|UNKNOWN|N/?A)?\s*$"


def naics_status(col):
    code = F.trim(F.split(F.coalesce(col, F.lit("")), r"\|").getItem(0))
    desc = F.upper(F.trim(F.split(F.coalesce(col, F.lit("")), r"\|").getItem(1)))
    return (
        F.when(F.trim(F.coalesce(col, F.lit(""))) == "", "missing")
        .when(code.rlike(NAICS_PLACEHOLDER_RE) | desc.isin("UNKNOWN", "UNK", ""), "placeholder")
        .otherwise("valid")
    )


def side(id_col, naics_col, is_src):
    return pkg.select(
        F.trim(F.col(id_col).cast("string")).alias("node_id"),
        naics_status(F.col(naics_col)).alias("naics_status"),
        F.col("amount").cast("double").alias("amt"),
        F.col("volume").cast("double").alias("vol"),
        F.lit(1 if is_src else 0).alias("as_src"),
        F.lit(0 if is_src else 1).alias("as_dst"),
    )


pkg_nodes = (
    side("source", "source_naics", True)
    .unionByName(side("dest", "dest_naics", False))
    .groupBy("node_id")
    .agg(
        F.sum("amt").alias("amt_total"),
        F.sum("vol").alias("vol_total"),
        F.max("as_src").alias("is_src"),
        F.max("as_dst").alias("is_dst"),
        F.min("naics_status").alias("naics_status"),   # missing < placeholder < valid
    )
    .persist(StorageLevel.DISK_ONLY)
)
note("G1_pkg_nodes", pkg_nodes.count())
note("G1_pkg_dollars", float(pkg_nodes.agg(F.sum("amt_total")).collect()[0][0] or 0))

### G2 — identifier reconciliation

Are these the same identifier space at all? If PKG `source` is account-level or
hashed rather than a party-level MDM id, no normalisation will rescue the join and
this becomes a data-engineering ask.

In [ ]:
def id_shape(df, col, label):
    c = F.trim(F.col(col).cast("string"))
    return (
        df.select(
            F.lit(label).alias("side"),
            F.length(c).alias("id_len"),
            F.when(c.rlike(r"^\d+$"), "numeric")
             .when(c.rlike(r"^[A-Za-z0-9]+$"), "alnum")
             .otherwise("other").alias("charset"),
        )
        .groupBy("side", "id_len", "charset")
        .agg(F.count("*").alias("n"))
    )


idp = id_shape(work, "mdm_id", "mdm_id").unionByName(
    id_shape(pkg_nodes, "node_id", "pkg_node")
).orderBy("side", F.desc("n"))
show(idp, 40)
REPORT["G2_id_shape"] = [r.asDict() for r in idp.collect()]

In [ ]:
mdm_ids = work.select(F.col("mdm_id").alias("raw")).distinct().persist(StorageLevel.DISK_ONLY)
pkg_ids = pkg_nodes.select(F.col("node_id").alias("raw")).distinct().persist(StorageLevel.DISK_ONLY)
n_pkg_ids = pkg_ids.count()

NORMS = {
    "raw":         lambda c: c,
    "upper":       lambda c: F.upper(c),
    "strip_zeros": lambda c: F.regexp_replace(c, r"^0+", ""),
    "digits_only": lambda c: F.regexp_replace(c, r"[^0-9]", ""),
    "upper_alnum": lambda c: F.regexp_replace(F.upper(c), r"[^A-Z0-9]", ""),
    "zfill18":     lambda c: F.lpad(F.regexp_replace(c, r"^0+", ""), 18, "0"),
}

rows = []
for name, fn in NORMS.items():
    a = mdm_ids.select(fn(F.col("raw")).alias("k")).distinct()
    b = pkg_ids.select(fn(F.col("raw")).alias("k")).distinct()
    m = b.join(a, "k", "left_semi").count()
    rows.append({"norm": name, "matched": m, "pct": round(100 * m / max(n_pkg_ids, 1), 3)})
    print(f"  {name:14s} {m:>12,}  {rows[-1]['pct']:>7.3f}%")

REPORT["G2b_join_normalisations"] = rows
BEST_NORM = max(rows, key=lambda r: r["matched"])["norm"]
BEST_PCT = max(r["pct"] for r in rows)
note("G2b_best_norm", BEST_NORM)
note("G2b_best_match_pct", BEST_PCT)

if BEST_PCT < 50:
    print("\n  !! Under half of PKG nodes resolve to an MDM party under any normalisation.\n"
          "  These are probably different identifier spaces (account vs party, or one\n"
          "  side hashed). Stop here and confirm the intended join path.")

In [ ]:
norm_fn = NORMS[BEST_NORM]

geo_party = work.select(
    norm_fn(F.col("mdm_id")).alias("join_key"),
    "has_geo", "state", "zip3", "coord_dp", "rec_type", "addr_type",
)
joined = (
    pkg_nodes.withColumn("join_key", norm_fn(F.col("node_id")))
    .join(geo_party, "join_key", "left")
    .persist(StorageLevel.DISK_ONLY)
)

In [ ]:
cov = joined.agg(
    F.count("*").alias("pkg_nodes"),
    F.sum(F.when(F.col("has_geo").isNotNull(), 1).otherwise(0)).alias("matched_to_mdm"),
    F.sum(F.coalesce(F.col("has_geo"), F.lit(0))).alias("with_coords"),
    F.sum("amt_total").alias("dollars_total"),
    F.sum(F.when(F.col("has_geo") == 1, F.col("amt_total")).otherwise(0.0)).alias("dollars_geocoded"),
    F.sum("vol_total").alias("volume_total"),
    F.sum(F.when(F.col("has_geo") == 1, F.col("vol_total")).otherwise(0.0)).alias("volume_geocoded"),
).collect()[0].asDict()

cov["pct_nodes_matched"] = round(100 * cov["matched_to_mdm"] / max(cov["pkg_nodes"], 1), 2)
cov["pct_nodes_geocoded"] = round(100 * cov["with_coords"] / max(cov["pkg_nodes"], 1), 2)
cov["pct_dollars_geocoded"] = round(100 * cov["dollars_geocoded"] / max(cov["dollars_total"], 1), 2)
cov["pct_volume_geocoded"] = round(100 * cov["volume_geocoded"] / max(cov["volume_total"], 1), 2)
print(json.dumps(cov, indent=2, default=str))
REPORT["G3_coverage"] = cov

note("G3_mdm_parties_not_in_pkg",
     geo_party.select("join_key").distinct()
     .join(joined.select("join_key").distinct(), "join_key", "left_anti").count())

**Reading G3.**

- dollars > nodes → coverage concentrated in large customers; the modules are more
  trustworthy than the raw node rate suggests. Brief the dollar number.
- dollars < nodes → geocoding is best on small nodes; the modules are weaker than
  they look and F4 becomes urgent.
- `pct_nodes_matched` well above `pct_nodes_geocoded` → the parties are there, they
  just have no usable coordinate. That is an imputation problem (M3), and tractable.
- `pct_nodes_matched` low → an identifier or entity-scope problem, and not.

In [ ]:
geo_pop = (
    F.when(F.col("has_geo").isNull(), "not_in_mdm")
    .when(F.col("has_geo") == 1, "geo_present")
    .otherwise("geo_missing")
)

g4 = (
    joined.withColumn("geo_pop", geo_pop)
    .groupBy("geo_pop")
    .agg(
        F.count("*").alias("n_nodes"),
        F.round(F.sum("amt_total") / 1e6, 1).alias("amt_musd"),
        F.round(F.expr("percentile_approx(amt_total, 0.5)"), 1).alias("median_amt"),
        F.round(F.avg("vol_total"), 2).alias("mean_vol"),
        F.round(F.avg("is_src"), 3).alias("frac_payer"),
        F.round(F.avg("is_dst"), 3).alias("frac_payee"),
    )
    .orderBy("geo_pop")
)
show(g4)
REPORT["G4_missingness_informative"] = [r.asDict() for r in g4.collect()]

g5 = (
    joined.withColumn("geo_pop", geo_pop)
    .groupBy("naics_status", "geo_pop")
    .agg(F.count("*").alias("n_nodes"), F.round(F.sum("amt_total") / 1e6, 1).alias("amt_musd"))
    .orderBy("naics_status", "geo_pop")
)
show(g5, 30)
REPORT["G5_geo_x_naics"] = [r.asDict() for r in g5.collect()]

**Why G5 matters.** If geo-missing and NAICS-placeholder are the same nodes, there is
one enrichment gap and one fix closes both. If they are independent, the imputation
ladder can lean on NAICS as a conditioning prior — which it cannot do if the two are
collinear.

### G6 — F4: the Pittsburgh test

Coverage-versus-distance cannot be computed from coordinates, because the nodes of
interest are exactly the ones without them. ZIP3 is the location proxy — it survives
when lat/lon does not — placed by the median coordinate of the geocoded rows sharing
it. Self-referential, but valid: it locates the non-geocoded rows without using their
own missing geocode.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    dlat, dlon = F.radians(lat2 - lat1), F.radians(lon2 - lon1)
    a = F.pow(F.sin(dlat / 2), 2) + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin(dlon / 2), 2)
    return F.lit(6371.0088) * 2 * F.asin(F.sqrt(F.least(a, F.lit(1.0))))


zip3_ref = (
    work.filter((F.col("has_geo") == 1) & F.col("zip3").rlike(r"^\d{3}$"))
    .groupBy("zip3")
    .agg(
        F.expr("percentile_approx(lat, 0.5)").alias("z_lat"),
        F.expr("percentile_approx(lon, 0.5)").alias("z_lon"),
        F.count("*").alias("n_ref"),
    )
    .filter(F.col("n_ref") >= 20)
    .withColumn("km_from_pit", haversine_km(F.col("z_lat"), F.col("z_lon"), F.lit(PIT_LAT), F.lit(PIT_LON)))
    .select("zip3", "km_from_pit")
    .persist(StorageLevel.DISK_ONLY)
)
note("G6_zip3_reference_points", zip3_ref.count())

In [ ]:
ring = (
    F.when(F.col("km_from_pit").isNull(), "9_no_zip3")
    .when(F.col("km_from_pit") < 50, "0_under_50km")
    .when(F.col("km_from_pit") < 150, "1_50_150km")
    .when(F.col("km_from_pit") < 400, "2_150_400km")
    .when(F.col("km_from_pit") < 1000, "3_400_1000km")
    .otherwise("4_over_1000km")
)

g6 = (
    joined.join(F.broadcast(zip3_ref), "zip3", "left")
    .withColumn("ring", ring)
    .groupBy("ring")
    .agg(
        F.count("*").alias("pkg_nodes"),
        F.round(100 * F.avg(F.coalesce(F.col("has_geo"), F.lit(0))), 2).alias("pct_geocoded"),
        F.sum("amt_total").alias("_amt"),
    )
    .withColumn("amt_musd", F.round(F.col("_amt") / 1e6, 1))
    .withColumn("pct_of_dollars", F.round(100 * F.col("_amt") / F.sum("_amt").over(Window.partitionBy()), 2))
    .drop("_amt")
    .orderBy("ring")
)
show(g6)
REPORT["G6_pittsburgh_rings"] = [r.asDict() for r in g6.collect()]

**Reading G6.** Two columns, two questions.

`pct_of_dollars` in ring `0_under_50km` is the actual test of the Pittsburgh belief,
on the graph-visible dollar-weighted population — the only version worth a deck.

`pct_geocoded` declining with distance is the coverage artifact. If local
relationships are older and branch-originated, their geocodes are cleaner, and any
node-count map exaggerates local concentration by construction. If that gradient is
present, every downstream geographic statistic needs a coverage-adjusted twin.

## H. Build the extract

One address per party means no `addr_rank` and no selection rule. Raw `addr_line_*`
text stays out of the output — the derived flags and `addr_norm_hash` preserve
everything the placeholder screen needs at a much smaller PII surface.

In [ ]:
geo_status = (
    F.when(F.col("has_geo") == 0, "missing")
    .when(F.col("rec_type").isin("POSTOFFICEBOX", "GENERALDELIVERY"), "placeholder")
    .when(F.col("flag_po_box") == 1, "placeholder")
    .when(F.col("coord_dp") <= LOW_PRECISION_DP, "low_precision")
    .otherwise("valid")
)

ext = (
    work.withColumn("geo_status", geo_status)
    .withColumn("shared_structure", F.when(F.col("rec_type") == "HIGHRISE", 1).otherwise(0))
    .withColumn("coord_key", F.when(F.col("has_geo") == 1, coord_key))
    .withColumn("snapshot", F.lit(str(MAX_SNAP)))
    .drop("has_geo")
)
show(ext.limit(10), 10, truncate=24)

node_geo = (
    ext.withColumn("join_key", norm_fn(F.col("mdm_id")))
    .join(
        pkg_nodes.withColumn("join_key", norm_fn(F.col("node_id")))
        .select("join_key", F.col("node_id").alias("pkg_node_id"),
                "amt_total", "vol_total", "is_src", "is_dst", "naics_status"),
        "join_key",
        "inner",
    )
    .drop("join_key")
    .persist(StorageLevel.DISK_ONLY)
)

n_node = node_geo.count()
n_dupe = node_geo.groupBy("pkg_node_id").count().filter(F.col("count") > 1).count()
note("H_node_rows", n_node)
note("H_duplicate_pkg_nodes", n_dupe)
assert n_dupe == 0, "pkg_node_id is not unique — investigate before writing."

show(
    node_geo.groupBy("geo_status")
    .agg(F.count("*").alias("n_nodes"), F.round(F.sum("amt_total") / 1e6, 1).alias("amt_musd"))
    .orderBy("geo_status")
)
REPORT["H_geo_status_mix"] = [
    r.asDict() for r in node_geo.groupBy("geo_status").count().collect()
]

In [ ]:
os.makedirs(OUT_DIR_LOCAL, exist_ok=True)

if WORK_DIR:
    node_geo.write.mode("overwrite").parquet(f"{WORK_DIR}/pkg_geo_node_{MAX_SNAP}")
    ext.write.mode("overwrite").parquet(f"{WORK_DIR}/pkg_geo_address_all_{MAX_SNAP}")
    print("parquet written under", WORK_DIR)

MAX_DRIVER_ROWS = 15_000_000
if n_node <= MAX_DRIVER_ROWS:
    pdf = node_geo.toPandas()
    # 18-digit ids lose precision as float64; zip5 loses its leading zero (08861 -> 8861).
    for c in ["mdm_id", "mdm_address_id", "pkg_node_id", "zip5", "zip4", "zip3", "addr_norm_hash"]:
        if c in pdf.columns:
            pdf[c] = pdf[c].astype("string")
    out_csv = f"{OUT_DIR_LOCAL}/pkg_geo_node_{MAX_SNAP}.csv"
    pdf.to_csv(out_csv, index=False)
    print(f"wrote {len(pdf):,} rows -> {out_csv}")
    print(pdf.dtypes)
elif WORK_DIR:
    node_geo.write.mode("overwrite").option("header", True).csv(f"{WORK_DIR}/pkg_geo_node_csv")
    print("too large for the driver; partitioned CSV written to WORK_DIR")
else:
    raise RuntimeError(f"{n_node:,} rows exceeds MAX_DRIVER_ROWS and WORK_DIR is unset.")

## I. Report

In [ ]:
print(json.dumps(REPORT, indent=2, default=str))
with open(f"{OUT_DIR_LOCAL}/geo_profile_report_v2_{MAX_SNAP}.json", "w") as fh:
    json.dump(REPORT, fh, indent=2, default=str)
print("report written")

---

## What I need back

1. **`G2b_join_normalisations` and `G3_coverage`** — everything is contingent on
   these. A low match rate makes the next conversation about identifiers, not
   geography.
2. **`B1_addr_type`** — with one address per party, this is the representativeness
   test. A book dominated by MAILING addresses is a materially different starting
   point from one dominated by PHYSICAL.
3. **`C1_coord_state`** — sizes of `4_non_numeric`, `5_null_island`, and
   `7_positive_lon_dropped_sign`.
4. **`F2_changes_by_snapshot`** — the whole table. The `regeocode` column identifies
   the vendor refresh dates; the `candidate_relocation` baseline sets the ceiling on
   how much signal geo dynamics can carry.
5. **`G6_pittsburgh_rings`** — both columns.
6. **`D1_top_clusters` / `D2_top_shared_addresses`** — top 20 each. The big ones
   usually explain themselves at a glance.

## Open decisions

`low_precision` is now its own `geo_status` value rather than being pooled with
placeholders, because ≤2 decimals and a PO box fail differently: one is a real address
at coarse resolution, the other is not the customer's location at all. If C2 shows the
low-precision population is tiny, collapse it back into `valid`; if it is large, it
needs its own handling policy in every downstream module.

If `F2` shows the `regeocode` share dominating, the address history is mostly vendor
churn and the dynamics module should be scoped to `candidate_relocation` events only —
which may be few enough to inspect by hand, and that would be a good outcome.